In [1]:
import pandas as pd

/Users/juliasbardelatti/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df_muni = pd.read_csv("./municipios_regioes.csv")

In [3]:
df = pd.read_excel("./malaria_2008.xlsx")
df.columns = ['uf','codigo_municipio_ibge_novo', 'municipio', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023']

In [4]:
df_residente = pd.read_csv("./populacao_residente.csv", sep=";")
df_residente[['codigo_municipio', 'nome_municipio']] = df_residente['Município'].str.split(' ', n=1, expand=True)
df_residente.columns = ['muni', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', 'codigo_municipio_ibge_novo', 'no_municipio']

In [5]:
df_merged = pd.merge(df, df_muni, on=['codigo_municipio_ibge_novo', 'uf'], how='inner')
df_merged = df_merged[['codigo_municipio_ibge', 'municipio', 'nome_uf', 'uf', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016','2017', '2018', '2019', '2020', '2021', '2022', '2023']]

estados_amazonia = ["AC", "AP", "AM", "MA", "MT", "PA", "RO", "RR", "TO"]
df_merged = df_merged[df_merged['uf'].isin(estados_amazonia)]

df_long = df_merged.melt(id_vars=['codigo_municipio_ibge', 'municipio', 'nome_uf'], 
                  value_vars=[str(ano) for ano in range(2008, 2024)],
                  var_name='ano',
                  value_name='valor')


df_long[['valor']] = df_long[['valor']].fillna(value=0)

df_long['ano'] = pd.to_numeric(df_long['ano'], errors='coerce')

In [6]:
df_residente['codigo_municipio_ibge_novo'] = pd.to_numeric(df_residente['codigo_municipio_ibge_novo'], errors='coerce')
df_muni['codigo_municipio_ibge_novo'] = pd.to_numeric(df_muni['codigo_municipio_ibge_novo'], errors='coerce')

df_merged_res = pd.merge(df_residente, df_muni, on=['codigo_municipio_ibge_novo'], how='inner')
df_merged_res = df_merged_res[['codigo_municipio_ibge', 'nome_municipio', 'nome_uf', 'uf', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016','2017', '2018', '2019', '2020', '2021', '2022', '2023']]

estados_amazonia = ["AC", "AP", "AM", "MA", "MT", "PA", "RO", "RR", "TO"]
df_merged_res = df_merged_res[df_merged_res['uf'].isin(estados_amazonia)]

df_long_res = df_merged_res.melt(id_vars=['codigo_municipio_ibge', 'nome_municipio', 'nome_uf'], 
                  value_vars=[str(ano) for ano in range(2008, 2024)],
                  var_name='ano',
                  value_name='pop')


df_long_res[['pop']] = df_long_res[['pop']].fillna(value=0)

df_long_res['ano'] = pd.to_numeric(df_long_res['ano'], errors='coerce')

In [11]:
df_malaria_pop = pd.merge(df_long_res, df_long, on=['codigo_municipio_ibge', 'nome_uf', 'ano'], how='inner')
df_malaria_pop.head()

,codigo_municipio_ibge,nome_municipio,nome_uf,ano,pop,municipio,valor
0,1100015,Alta Floresta D'oeste,Rondônia,2008,25440,Alta Floresta D'Oeste,210.0
1,1100023,Ariquemes,Rondônia,2008,88293,Ariquemes,1156.0
2,1100031,Cabixi,Rondônia,2008,6687,Cabixi,0.0
3,1100049,Cacoal,Rondônia,2008,78105,Cacoal,31.0
4,1100056,Cerejeiras,Rondônia,2008,17598,Cerejeiras,4.0


In [8]:
df_malaria_pop.to_csv("dados_malaria.csv")